# <b> RETO I  </b>
## <b> ADRIAN VAZQUEZ </b>

---

# Physics-Informed Neural Networks (PINNs) para la Ecuación de Burgers

## Objetivo

El objetivo de este ejercicio es implementar una **Physics-Informed Neural Network (PINN)** para aproximar la solución de la ecuación viscosa de Burgers, una ecuación diferencial parcial (PDE) no lineal ampliamente utilizada en dinámica de fluidos, propagación de ondas y otros problemas de física matemática.

A diferencia de las redes neuronales tradicionales, una PINN no aprende únicamente a partir de datos observados. En su lugar, incorpora directamente las leyes físicas que gobiernan el sistema dentro de la función de pérdida, obligando a la red a producir soluciones consistentes con la ecuación diferencial.

La PDE considerada es:

$$
\frac{\partial u}{\partial t}
+
u\frac{\partial u}{\partial x}
=
\nu \frac{\partial^2 u}{\partial x^2}
$$

donde:

- $u(x,t)$ representa la variable de estado del sistema.
- $x$ es la coordenada espacial.
- $t$ es el tiempo.
- $\nu$ es el coeficiente de viscosidad.

---

## Idea General de una PINN

Buscamos aproximar la solución desconocida

$$
u(x,t)
$$

mediante una red neuronal

$$
u_{\theta}(x,t)
$$

parametrizada por los pesos y sesgos $\theta$.

La principal ventaja de una PINN es que las derivadas requeridas por la PDE se obtienen mediante **Automatic Differentiation (Autograd)**, permitiendo construir el residuo físico:

$$
R(x,t)
=
\frac{\partial u_{\theta}}{\partial t}
+
u_{\theta}
\frac{\partial u_{\theta}}{\partial x}
-
\nu
\frac{\partial^2 u_{\theta}}{\partial x^2}
$$

Si la red representa correctamente la solución de la PDE, entonces:

$$
R(x,t) \approx 0
$$

para todos los puntos del dominio.

---

## Función de Pérdida

La función de pérdida estará compuesta por dos términos principales:

### 1. Residuo de la PDE

Penaliza cualquier violación de la ecuación diferencial:

$$
\mathcal{L}_{PDE}
=
\mathbb{E}
\left[
R(x,t)^2
\right]
$$

### 2. Condición Inicial

Garantiza que la solución satisfaga la condición inicial especificada:

$$
\mathcal{L}_{IC}
=
\mathbb{E}
\left[
\left(
u_{\theta}(x,0)
-
u_0(x)
\right)^2
\right]
$$

### Pérdida Total

$$
\mathcal{L}
=
\mathcal{L}_{PDE}
+
\mathcal{L}_{IC}
$$

---

## Metodología

El flujo general del ejercicio será:

1. Definir el dominio espacial y temporal.
2. Construir una red neuronal totalmente conectada (MLP).
3. Utilizar Automatic Differentiation para calcular:
   - $u_t$
   - $u_x$
   - $u_{xx}$
4. Construir el residuo de Burgers.
5. Definir la función de pérdida física.
6. Entrenar la PINN mediante descenso de gradiente.
7. Visualizar la solución aprendida.
8. Analizar la convergencia del residuo y de la pérdida total.

---

## Relación con Black-Scholes

Este problema es conceptualmente similar al enfoque utilizado para resolver la ecuación de Black-Scholes mediante PINNs.

En ambos casos:

- Existe una PDE que describe la dinámica del sistema.
- La red neuronal aproxima una función desconocida.
- Las derivadas se calculan mediante diferenciación automática.
- El entrenamiento minimiza el incumplimiento de la ecuación diferencial.

Por esta razón, la ecuación de Burgers suele utilizarse como un problema introductorio antes de abordar aplicaciones más avanzadas en finanzas cuantitativas, como la valuación de derivados mediante PINNs.

In [1]:
# 1. Imports y configuración inicial


import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

# Dispositivo: GPU si está disponible, si no CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

Using device: cpu


# <b> 2. Arquitectura de la PINN </b>

In [2]:
# 2. Definición de la PINN

class PINN(nn.Module):

    def __init__(self,
                 input_dim=2,
                 hidden_dim=50,
                 num_hidden_layers=4,
                 output_dim=1):

        super().__init__()

        layers = []

        # Capa de entrada
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.Tanh())

        # Capas ocultas
        for _ in range(num_hidden_layers):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())

        # Capa de salida
        layers.append(nn.Linear(hidden_dim, output_dim))

        self.network = nn.Sequential(*layers)

    def forward(self, x, t):

        inputs = torch.cat([x, t], dim=1)

        return self.network(inputs)

# <b> 3. Inicializar el modelo </b>

In [3]:
# 3. Crear modelo
model = PINN().to(device)
print(model)

PINN(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=50, bias=True)
    (1): Tanh()
    (2): Linear(in_features=50, out_features=50, bias=True)
    (3): Tanh()
    (4): Linear(in_features=50, out_features=50, bias=True)
    (5): Tanh()
    (6): Linear(in_features=50, out_features=50, bias=True)
    (7): Tanh()
    (8): Linear(in_features=50, out_features=50, bias=True)
    (9): Tanh()
    (10): Linear(in_features=50, out_features=1, bias=True)
  )
)


# <b> 4.Generación de: puntos interiores de la PDE y puntos de condición inicial </b>

que son los dos conjuntos de datos que alimentarán la pérdida

$ L=L_{PDE} +L_{IC} $

## <b> 4.1 Dominio y datos de entrenamiento </b>

Usaremos:

$ x∈[−1,1],t∈[0,1] $

y una condición inicial típica:

$ u(x,0)=−sin(πx) $
	​

In [4]:
# 4. Dominio y puntos de entrenamiento

# Parámetro de viscosidad
nu = 0.01 / np.pi

# Número de puntos
N_f = 10_000   # puntos interiores para la PDE
N_ic = 200     # puntos para condición inicial

# Dominio
x_min, x_max = -1.0, 1.0
t_min, t_max = 0.0, 1.0

# Puntos interiores: PDE
x_f = x_min + (x_max - x_min) * torch.rand((N_f, 1), device=device)
t_f = t_min + (t_max - t_min) * torch.rand((N_f, 1), device=device)

# Necesitamos gradientes respecto a x y t
x_f.requires_grad_(True)
t_f.requires_grad_(True)


# Condición inicial: t = 0
x_ic = x_min + (x_max - x_min) * torch.rand((N_ic, 1), device=device)
t_ic = torch.zeros((N_ic, 1), device=device)

u_ic = -torch.sin(np.pi * x_ic)

print("Puntos PDE:", x_f.shape, t_f.shape)
print("Puntos IC:", x_ic.shape, t_ic.shape, u_ic.shape)

Puntos PDE: torch.Size([10000, 1]) torch.Size([10000, 1])
Puntos IC: torch.Size([200, 1]) torch.Size([200, 1]) torch.Size([200, 1])


Ahora seguimos con la función que calcula el residuo de Burgers:

$ R(x,t)=u_{t} +uu_{x} −νu_{xx} $ 

In [5]:
# 5. Residuo de la PDE de Burgers

def burgers_residual(model, x, t, nu):
    """
    Calcula el residuo:
    
        R(x,t) = u_t + u u_x - nu u_xx
    
    donde u = u_theta(x,t)
    """

    # Predicción de la red
    u = model(x, t)

    # Primera derivada respecto a t: u_t
    u_t = torch.autograd.grad(
        u, t,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True
    )[0]

    # Primera derivada respecto a x: u_x
    u_x = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True
    )[0]

    # Segunda derivada respecto a x: u_xx
    u_xx = torch.autograd.grad(
        u_x, x,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True,
        retain_graph=True
    )[0]

    # Residuo de Burgers
    residual = u_t + u * u_x - nu * u_xx

    return residual

In [6]:
residual_test = burgers_residual(model, x_f, t_f, nu)
print("Residual shape:", residual_test.shape)
print("Residual mean:", residual_test.mean().item())
print("Residual std:", residual_test.std().item())

Residual shape: torch.Size([10000, 1])
Residual mean: 0.04189944639801979
Residual std: 0.004616192542016506


In [7]:
#  Función de pérdida total


def pinn_loss(model, x_f, t_f, x_ic, t_ic, u_ic, nu):
    """
    Loss total de la PINN:
    
        L = L_PDE + L_IC
    
    donde:
    - L_PDE penaliza violaciones de la ecuación de Burgers
    - L_IC penaliza errores en la condición inicial
    """

    # Residuo físico
    residual = burgers_residual(model, x_f, t_f, nu)
    loss_pde = torch.mean(residual**2)

    # Condición inicial
    u_pred_ic = model(x_ic, t_ic)
    loss_ic = torch.mean((u_pred_ic - u_ic)**2)

    # Loss total
    loss = loss_pde + loss_ic

    return loss, loss_pde, loss_ic

In [8]:
loss, loss_pde, loss_ic = pinn_loss(model, x_f, t_f, x_ic, t_ic, u_ic, nu)

print("Total loss:", loss.item())
print("PDE loss:", loss_pde.item())
print("IC loss:", loss_ic.item())

Total loss: 0.5052080154418945
PDE loss: 0.0017768709221854806
IC loss: 0.5034311413764954
